# CartPole 맛보기 실습

이번 실습의 목표는 다음과 같습니다.

1. CartPole 환경의 **상태(state)**, **행동(action)**, **보상(reward)** 을 이해한다.
2. **랜덤 정책**, **규칙 기반 정책**, **나만의 정책**을 비교한다.
3. 상태 정보를 어떻게 사용하느냐에 따라 성능이 달라질 수 있음을 확인한다.
4. 마지막으로 **DQN 데모**를 보고 Deep RL이 왜 필요한지 연결한다.

## 실습 안내

- 이번 실습은 **빈칸(TODO)** 을 채우는 방식으로 진행합니다.
- 코드를 완전히 새로 작성하기보다, **핵심 부분만 직접 구현**합니다.
- 실습의 목적은 정답 코드 작성 자체보다, **왜 이런 정책이 작동하는지 생각해보는 것**입니다.
- 수식은 모두 `$$ ... $$` 형태로 넣어 두어 노트북에서 안정적으로 보이도록 했습니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Video
import random

try:
    import gymnasium as gym
    GYM_BACKEND = "gymnasium"
except ImportError:
    try:
        import gym
        GYM_BACKEND = "gym"
    except ImportError as e:
        raise ImportError(
            "gymnasium 또는 gym 패키지가 필요합니다. "
            "실습 환경에 맞는 패키지를 먼저 설치해주세요."
        ) from e

np.random.seed(30)
CartPole_MAX_STEPS = 500

def make_env(render_mode=None):
    try:
        if render_mode is None:
            return gym.make("CartPole-v1", max_episode_steps=CartPole_MAX_STEPS)
        return gym.make("CartPole-v1", render_mode=render_mode, max_episode_steps=CartPole_MAX_STEPS)
    except TypeError:
        return gym.make("CartPole-v1", max_episode_steps=CartPole_MAX_STEPS)

def reset_env(env, seed=None):
    result = env.reset(seed=seed) if seed is not None else env.reset()
    if isinstance(result, tuple):
        obs, info = result
    else:
        obs, info = result, {}
    return np.array(obs, dtype=float), info

def step_env(env, action):
    result = env.step(action)
    if len(result) == 5:
        next_obs, reward, terminated, truncated, info = result
        done = terminated or truncated
    else:
        next_obs, reward, done, info = result
    return np.array(next_obs, dtype=float), float(reward), bool(done), info

## 1. CartPole 환경 이해하기

CartPole은 **막대가 쓰러지지 않도록 카트를 좌우로 움직이는 문제**입니다.

- **상태(state)**: 현재 카트와 막대의 상태를 나타내는 정보
- **행동(action)**: 왼쪽(0) 또는 오른쪽(1)으로 힘을 주는 선택
- **보상(reward)**: 한 step 버티면 보통 1점을 받음
- **목표(goal)**: 가능한 오래 막대를 세우기

강화학습 관점에서 보면, CartPole은 매우 단순해 보이지만  
**상태가 연속값**이라는 점 때문에 나중에 Deep RL과도 자연스럽게 연결됩니다.

In [ ]:
# TODO: CartPole 환경을 생성해봅시다.
env = make_env()

# 초기 상태를 확인해봅시다.
obs, info = reset_env(env, seed=42)

print("초기 상태(observation) =", obs)
print("상태 벡터 길이 =", len(obs))
print("가능한 행동 수 =", env.action_space.n)
print(f"수정된 최대 스텝: {env.spec.max_episode_steps}")
print("사용 중인 backend =", GYM_BACKEND)


### CartPole 상태는 무엇을 의미할까?

CartPole의 관측값은 보통 4개의 값으로 이루어집니다.

1. **카트 위치** (cart position)
2. **카트 속도** (cart velocity)
3. **막대 각도** (pole angle)
4. **막대 각속도** (pole angular velocity)

즉, 상태는 단일 숫자가 아니라  
**현재 상황을 표현하는 여러 정보의 묶음**이라고 볼 수 있습니다.

In [ ]:
# 한 step만 직접 실행해보면서 상태, 보상, 종료 여부를 확인해봅시다.
action = 0  # 0: 왼쪽, 1: 오른쪽
next_obs, reward, done, info = step_env(env, action)

print("선택한 행동 =", action)
print("다음 상태   =", next_obs)
print("보상        =", reward)
print("종료 여부   =", done)

## 2. 랜덤 정책 실행하기

가장 단순한 정책은 **매 step마다 무작위로 행동**하는 것입니다.

질문:
- 아무 생각 없이 좌/우를 선택하면 얼마나 오래 버틸 수 있을까요?
- 이 결과를 기준선(baseline)으로 삼아, 이후 정책들과 비교해봅시다.

In [ ]:
# MAX_STEPS = 1000
def run_episode(env, policy_fn, seed=None, max_steps=CartPole_MAX_STEPS):
    obs, info = reset_env(env, seed=seed)
    total_reward = 0.0
    trajectory = [obs.copy()]

    for step in range(max_steps):
        action = int(policy_fn(obs))
        next_obs, reward, done, info = step_env(env, action)

        total_reward += reward
        trajectory.append(next_obs.copy())
        obs = next_obs

        if done:
            break

    episode_length = len(trajectory) - 1
    return total_reward, episode_length, np.array(trajectory)

def evaluate_policy(policy_fn, n_episodes=30, seed_start=0):
    env = make_env()
    rewards, lengths = [], []

    for i in range(n_episodes):
        total_reward, episode_length, _ = run_episode(
            env, policy_fn, seed=seed_start + i
        )
        rewards.append(total_reward)
        lengths.append(episode_length)

    env.close()
    return np.array(rewards), np.array(lengths)

def summarize_result(name, rewards, lengths):
    print(f"[{name}]")
    print(f"- 평균 보상   : {rewards.mean():.2f}")
    print(f"- 평균 길이   : {lengths.mean():.2f}")
    print(f"- 최대 길이   : {lengths.max():.0f}")
    print(f"- 최소 길이   : {lengths.min():.0f}")

In [ ]:
def random_policy(obs):
    '''
    랜덤 정책:
    상태를 보지 않고 0 또는 1을 무작위로 선택합니다.
    '''
    # TODO: 랜덤하게 0 또는 1을 반환하세요.
    # raise NotImplementedError("TODO: random_policy를 완성하세요.")
    return random.randint(0, 1)

In [ ]:
# 랜덤 정책을 여러 번 실행해봅시다.
random_rewards, random_lengths = evaluate_policy(random_policy, n_episodes=30, seed_start=0)
summarize_result("Random Policy", random_rewards, random_lengths)

plt.figure(figsize=(8, 4))
plt.plot(random_lengths, marker="o")
plt.title("Random Policy: Episode Length")
plt.xlabel("Episode")
plt.ylabel("Length")
plt.grid(alpha=0.3)
plt.show()

## 3. 규칙 기반 정책 만들기

이제 사람이 직접 간단한 규칙을 만들어봅시다.

가장 단순한 아이디어는 다음과 같습니다.

- 막대가 **왼쪽으로 기울면 왼쪽으로 이동**
- 막대가 **오른쪽으로 기울면 오른쪽으로 이동**

즉, **막대 각도(pole angle)** 만 보고 행동을 선택하는 정책입니다.

In [ ]:
def rule_policy_angle_only(obs):
    '''
    막대 각도만 사용하는 규칙 기반 정책
    obs = [cart_position, cart_velocity, pole_angle, pole_angular_velocity]
    '''
    # TODO 1: pole_angle 값을 꺼내세요.
    # 힌트: pole_angle은 obs의 3번째 값입니다.
    # raise NotImplementedError("TODO: pole_angle을 사용한 규칙을 완성하세요.")
    pole_angle = obs[2]
    # TODO 2:
    # 막대가 왼쪽으로 기울면 0(왼쪽), 오른쪽으로 기울면 1(오른쪽)을 반환해보세요.
    if pole_angle < 0:
        return 0
    else:
        return 1

In [ ]:
angle_rewards, angle_lengths = evaluate_policy(rule_policy_angle_only, n_episodes=30, seed_start=100)
summarize_result("Rule Policy (Angle Only)", angle_rewards, angle_lengths)

plt.figure(figsize=(8, 4))
plt.plot(random_lengths, marker="o", label="Random")
plt.plot(angle_lengths, marker="s", label="Angle Only")
plt.title("Random vs Rule Policy (Angle Only)")
plt.xlabel("Episode")
plt.ylabel("Length")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 4. 상태를 일부만 쓸 때와 더 많이 쓸 때 비교하기

각도만 보는 정책은 단순하지만 한계가 있습니다.

이번에는 다음 정보를 함께 사용해봅시다.

- 막대 각도 (pole angle)
- 막대 각속도 (pole angular velocity)

질문:
- 각도만 보는 것보다 각속도까지 함께 보면 왜 더 좋아질 수 있을까요?
- 상태를 더 잘 표현하면 정책이 더 좋아질 수 있을까요?

In [ ]:
def rule_policy_angle_velocity(obs):
    '''
    막대 각도 + 각속도를 함께 사용하는 규칙 기반 정책
    '''
    # TODO 1: pole_angle과 pole_angular_velocity를 꺼내세요.
    # 힌트: obs[2], obs[3]
    # raise NotImplementedError("TODO: angle + angular velocity 정책을 완성하세요.")
    cart_position = obs[0]
    cart_velocity = obs[1]
    pole_angle = obs[2]
    pole_angular_velocity = obs[3]

    # TODO 2:
    # 예시 아이디어: score = pole_angle + 0.5 * pole_angular_velocity
    # score가 음수면 왼쪽(0), 양수면 오른쪽(1)으로 가게 해보세요.
    score = pole_angle +  pole_angular_velocity
    if score < 0:
        return 0
    else:
        return 1


In [ ]:
angle_vel_rewards, angle_vel_lengths = evaluate_policy(
    rule_policy_angle_velocity, n_episodes=30, seed_start=200
)
summarize_result("Rule Policy (Angle + Angular Velocity)", angle_vel_rewards, angle_vel_lengths)

plt.figure(figsize=(8, 4))
plt.plot(angle_lengths, marker="o", label="Angle Only")
plt.plot(angle_vel_lengths, marker="s", label="Angle + Angular Velocity")
plt.title("Rule Policy Comparison")
plt.xlabel("Episode")
plt.ylabel("Length")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 5. 나만의 규칙 기반 정책 만들기

이제 여러분만의 정책을 직접 만들어봅시다.

사용할 수 있는 상태 정보:
- 카트 위치
- 카트 속도
- 막대 각도
- 막대 각속도

생각해볼 질문:
- 어떤 상태 정보를 사용할 것인가?
- 임계값(threshold)을 둘 것인가?
- 각도만 볼 것인가, 속도도 함께 볼 것인가?
- 사람이 직접 규칙을 만드는 방식의 한계는 무엇일까?

In [ ]:
def my_policy(obs):
    '''
    TODO: 자신만의 규칙 기반 정책을 작성해보세요.
    '''
    # 예시 힌트:
    cart_position = obs[0]
    cart_velocity = obs[1]
    pole_angle = obs[2]
    pole_angular_velocity = obs[3]

    #print(pole_angle_score)
    score =  pole_angle + pole_angular_velocity  + cart_velocity
    if score < 0:
        return 0
    else:
        return 1

    raise NotImplementedError("TODO: 자신만의 정책을 작성하세요.")

In [ ]:
my_rewards, my_lengths = evaluate_policy(my_policy, n_episodes=30, seed_start=300)
summarize_result("My Policy", my_rewards, my_lengths)

plt.figure(figsize=(10, 4))
plt.plot(random_lengths, label="Random", alpha=0.7)
plt.plot(angle_lengths, label="Angle Only", alpha=0.7)
plt.plot(angle_vel_lengths, label="Angle + Angular Velocity", alpha=0.7)
plt.plot(my_lengths, label="My Policy", linewidth=2)
plt.title("Policy Comparison")
plt.xlabel("Episode")
plt.ylabel("Length")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 생각해보기

1. 내가 만든 규칙은 어떤 상태 정보를 사용했는가?
2. 왜 그 정보를 중요하다고 생각했는가?
3. 사람이 직접 정책을 만드는 방식의 장점과 한계는 무엇인가?

## 6. 왜 Q-table 방식이 어려울까?

CartPole은 보상이 단순합니다.  
하지만 상태는 다음처럼 **연속값**으로 주어집니다.

- 카트 위치
- 카트 속도
- 막대 각도
- 막대 각속도

이런 연속 상태를 모두 표 형태(Q-table)로 나누어 저장하려면  
상태 공간이 매우 커질 수 있습니다.

즉,

$$
\text{보상은 단순하지만, 상태는 연속적이다}
$$

그래서 CartPole에서는 나중에 **함수 근사(function approximation)** 와  
**Deep RL**이 필요해집니다.

In [ ]:
# 여러 step 동안 관측값을 모아보며 상태가 연속적으로 바뀌는지 확인해봅시다.
env = make_env()
obs, info = reset_env(env, seed=123)

samples = [obs.copy()]
for _ in range(10):
    action = env.action_space.sample()
    obs, reward, done, info = step_env(env, action)
    samples.append(obs.copy())
    if done:
        break

env.close()

samples = np.array(samples)
print("샘플 상태들:")
print(samples)

plt.figure(figsize=(10, 4))
for i, name in enumerate(["cart_pos", "cart_vel", "pole_angle", "pole_ang_vel"]):
    plt.plot(samples[:, i], marker="o", label=name)
plt.title("Sample Observations from CartPole")
plt.xlabel("Time step")
plt.ylabel("Value")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 7. DQN 데모 보기

지금까지는 사람이 직접 규칙 기반 정책을 만들어 보았습니다.  
이제는 **신경망으로 Q-value를 근사하는 DQN**이 어떻게 동작하는지, 그리고 왜 성능이 좋아질 수 있는지 확인해 보겠습니다.

이번 데모의 방향은 다음과 같습니다.

1. 작은 신경망을 만든다
2. CPU에서 짧게 DQN을 학습한다
3. 학습 곡선을 확인한다
4. 학습된 정책을 실제로 실행해 본다
5. 랜덤 정책 / 규칙 기반 정책 / DQN 정책을 비교해 본다

> **중요**  
> - 이 데모는 **GPU 없이 CPU만 사용**합니다.  
> - 수업에서는 강사가 미리 실행한 결과를 보여주고, 수강생은 나중에 직접 다시 돌려볼 수 있도록 구성했습니다.

### 7-1. 필요한 라이브러리와 실행 환경 준비

DQN 데모에서는 다음 요소가 필요합니다.

- **PyTorch**: 작은 신경망을 만들고 학습하기 위해 사용
- **Replay Buffer**: 이전 경험을 저장해 두었다가 미니배치로 꺼내 쓰기 위해 사용
- **Target Network**: 학습을 조금 더 안정적으로 만들기 위해 사용

이번 데모에서는 **CPU만 사용**하고, 실습 시간이 너무 길어지지 않도록 **작은 네트워크 + 적당한 episode 수**로 구성합니다.

In [ ]:
import random
from collections import deque

import torch
import torch.nn as nn
import torch.optim as optim
import imageio.v2 as imageio
from IPython.display import Image

# CPU만 사용합니다.
DEVICE = torch.device("cpu")
torch.set_num_threads(1)

# 재현성을 위해 seed를 고정합니다.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"사용 device: {DEVICE}")
print("이번 DQN 데모는 CPU만 사용합니다.")

### 7-2. 작은 DQN 모델과 Replay Buffer 만들기

DQN의 핵심 아이디어는 다음과 같습니다.

- 입력: 현재 상태 \(s\)
- 출력: 각 행동의 Q-value
- 행동 선택: 출력된 Q-value 중 가장 큰 행동 선택
- 학습 target:  
  $$
  y = r + \gamma \max_{a'} Q_{\text{target}}(s', a')
  $$

이번 데모에서는 다음과 같이 아주 작게 구성합니다.

- 상태 크기: 4
- 행동 개수: 2
- 은닉층: 작게 2개
- Replay Buffer: 최근 경험만 저장

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_dim=4, action_dim=2, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
        )

    def forward(self, x):
        return self.net(x)


class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((
            np.array(state, dtype=np.float32),
            int(action),
            float(reward),
            np.array(next_state, dtype=np.float32),
            float(done),
        ))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)

        states = torch.tensor(np.array(states), dtype=torch.float32, device=DEVICE)
        actions = torch.tensor(actions, dtype=torch.int64, device=DEVICE)
        rewards = torch.tensor(rewards, dtype=torch.float32, device=DEVICE)
        next_states = torch.tensor(np.array(next_states), dtype=torch.float32, device=DEVICE)
        dones = torch.tensor(dones, dtype=torch.float32, device=DEVICE)

        return states, actions, rewards, next_states, dones

    def __len__(self):
        return len(self.buffer)


def moving_average(x, window=10):
    x = np.asarray(x, dtype=float)
    if len(x) < window:
        return np.array([])
    return np.convolve(x, np.ones(window) / window, mode="valid")

### 7-3. 학습에 사용할 하이퍼파라미터와 보조 함수

이번 데모는 **짧게 학습해도 어느 정도 좋아지는 모습**을 보이는 것이 목표입니다.

따라서 하이퍼파라미터는 다음처럼 단순하게 둡니다.

- episode 수: 너무 크지 않게
- epsilon: 처음에는 탐험을 많이 하고, 점점 줄이기
- target network: 일정 episode마다 복사
- batch 학습: replay buffer에서 미니배치 추출

> 아래 숫자는 "정답"이 아니라, **CPU에서 무리 없이 돌릴 수 있는 예시 설정**입니다.

In [ ]:
# 하이퍼파라미터
# NUM_EPISODES = 160
# MAX_STEPS = 500
# GAMMA = 0.99
# LEARNING_RATE = 1e-3
# BATCH_SIZE = 64
# BUFFER_SIZE = 10000
# MIN_BUFFER_SIZE = 500
# EPSILON_START = 1.0
# EPSILON_END = 0.05
# EPSILON_DECAY = 0.97
# TARGET_UPDATE_EVERY = 10
NUM_EPISODES = 200
MAX_STEPS = 500

GAMMA = 0.99
LEARNING_RATE = 5e-4

BATCH_SIZE = 64
BUFFER_SIZE = 50000
MIN_BUFFER_SIZE = 1000

EPSILON_START = 1.0
EPSILON_END = 0.05
EPSILON_DECAY = 0.995

TARGET_UPDATE_STEPS = 500
EVAL_EVERY = 20

def select_action(state, online_net, epsilon, env):
    if random.random() < epsilon:
        return int(env.action_space.sample())

    state_tensor = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    with torch.no_grad():
        q_values = online_net(state_tensor)
    return int(torch.argmax(q_values, dim=1).item())


def optimize_model(online_net, target_net, replay_buffer, optimizer):
    if len(replay_buffer) < MIN_BUFFER_SIZE:
        return None

    states, actions, rewards, next_states, dones = replay_buffer.sample(BATCH_SIZE)

    q_values = online_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

    with torch.no_grad():
        # Double DQN:
        # action 선택은 online_net
        next_actions = online_net(next_states).argmax(dim=1)

        # 선택된 action의 Q-value 평가는 target_net
        next_q_values = target_net(next_states).gather(
            1, next_actions.unsqueeze(1)
        ).squeeze(1)

        targets = rewards + GAMMA * next_q_values * (1.0 - dones)

    loss = nn.SmoothL1Loss()(q_values, targets)

    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(online_net.parameters(), max_norm=10.0)
    optimizer.step()

    return float(loss.item())

In [ ]:
def evaluate_dqn_greedy(model, n_episodes=5, seed_start=9000):
    env = make_env()
    lengths = []

    for i in range(n_episodes):
        state, _ = reset_env(env, seed=seed_start + i)

        for step in range(MAX_STEPS):
            state_tensor = torch.tensor(
                state, dtype=torch.float32, device=DEVICE
            ).unsqueeze(0)

            with torch.no_grad():
                q_values = model(state_tensor)
                action = int(torch.argmax(q_values, dim=1).item())

            next_state, reward, done, _ = step_env(env, action)
            state = next_state

            if done:
                break

        lengths.append(step + 1)

    env.close()
    return float(np.mean(lengths))

### 7-4. DQN 학습 루프

이제 실제로 학습을 진행합니다.

한 episode 안에서는 다음 순서가 반복됩니다.

1. 현재 상태에서 행동 선택  
2. 환경에 action 적용  
3. \((s, a, r, s', done)\) 경험 저장  
4. replay buffer에서 batch를 꺼내 학습  
5. episode가 끝나면 epsilon 감소  
6. 일정 주기마다 target network 갱신

> 수업에서는 이 셀을 미리 실행해 두고 결과만 보여주면 됩니다.  
> 수강생은 나중에 episode 수나 hidden size를 바꿔 보며 실험해 볼 수 있습니다.

In [ ]:
def train_dqn(num_episodes=NUM_EPISODES, seed=SEED):
    env = make_env()

    online_net = QNetwork().to(DEVICE)
    target_net = QNetwork().to(DEVICE)
    target_net.load_state_dict(online_net.state_dict())

    optimizer = optim.Adam(online_net.parameters(), lr=LEARNING_RATE)
    replay_buffer = ReplayBuffer(capacity=BUFFER_SIZE)

    epsilon = EPSILON_START

    episode_rewards = []
    episode_lengths = []
    episode_losses = []
    eval_scores = []

    best_eval_score = -1
    best_state_dict = None

    global_step = 0

    for episode in range(num_episodes):
        state, _ = reset_env(env, seed=seed + episode)

        total_reward = 0.0
        losses_in_episode = []

        for step in range(MAX_STEPS):
            global_step += 1

            action = select_action(state, online_net, epsilon, env)
            next_state, reward, done, _ = step_env(env, action)

            replay_buffer.push(state, action, reward, next_state, done)

            loss = optimize_model(online_net, target_net, replay_buffer, optimizer)
            if loss is not None:
                losses_in_episode.append(loss)

            state = next_state
            total_reward += reward

            # target network를 step 기준으로 업데이트
            if global_step % TARGET_UPDATE_STEPS == 0:
                target_net.load_state_dict(online_net.state_dict())

            if done:
                break

        episode_rewards.append(total_reward)
        episode_lengths.append(step + 1)
        episode_losses.append(
            np.mean(losses_in_episode) if losses_in_episode else np.nan
        )

        epsilon = max(EPSILON_END, epsilon * EPSILON_DECAY)

        # 일정 episode마다 greedy policy 평가 후 best model 저장
        if (episode + 1) % EVAL_EVERY == 0:
            eval_score = evaluate_dqn_greedy(
                online_net,
                n_episodes=5,
                seed_start=9000 + episode * 10
            )
            eval_scores.append(eval_score)

            if eval_score > best_eval_score:
                best_eval_score = eval_score
                best_state_dict = {
                    k: v.cpu().clone()
                    for k, v in online_net.state_dict().items()
                }

            recent_mean = np.mean(episode_lengths[-10:])

            print(
                f"Episode {episode + 1:3d}/{num_episodes} | "
                f"epsilon={epsilon:.3f} | "
                f"최근 10개 평균 길이={recent_mean:.1f} | "
                f"greedy_eval={eval_score:.1f} | "
                f"best={best_eval_score:.1f}"
            )

    env.close()

    # 마지막 모델이 아니라 best model을 사용
    if best_state_dict is not None:
        online_net.load_state_dict(best_state_dict)

    return online_net, {
        "episode_rewards": np.array(episode_rewards, dtype=float),
        "episode_lengths": np.array(episode_lengths, dtype=float),
        "episode_losses": np.array(episode_losses, dtype=float),
        "eval_scores": np.array(eval_scores, dtype=float),
        "best_eval_score": best_eval_score,
    }

### 7-5. 학습 실행 및 학습 곡선 확인

아래 셀을 실행하면 DQN을 실제로 학습합니다.

학습이 끝나면 다음을 확인합니다.

- episode 길이가 점점 길어지는지
- 최근 평균이 올라가는지
- loss가 대체로 안정적으로 줄어드는지

> CartPole은 비교적 쉬운 환경이라, 작은 네트워크와 CPU만으로도  
> 어느 정도 성능 향상을 확인할 수 있습니다.

In [ ]:
dqn_model, dqn_history = train_dqn(num_episodes=NUM_EPISODES)

# 결과 저장: 나중에 다시 불러와 데모에 사용할 수 있습니다.
np.savez(
    "cartpole_dqn_demo_cpu.npz",
    episode_rewards=dqn_history["episode_rewards"],
    episode_lengths=dqn_history["episode_lengths"],
    episode_losses=dqn_history["episode_losses"],
)
torch.save(dqn_model.state_dict(), "cartpole_dqn_small_cpu.pt")

lengths = dqn_history["episode_lengths"]
losses = dqn_history["episode_losses"]
length_ma = moving_average(lengths, window=10)


fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(lengths, label="Episode Length", alpha=0.7)
if len(length_ma) > 0:
    axes[0].plot(
        np.arange(len(length_ma)) + 9,
        length_ma,
        label="Moving Average (window=10)",
        linewidth=2,
    )
axes[0].set_title("DQN score w.r.t Episode")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Length")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(losses, label="Loss", alpha=0.7)
axes[1].set_title("DQN : Loss")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Loss")
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

print("학습 결과를 cartpole_dqn_demo_cpu.npz, cartpole_dqn_small_cpu.pt 로 저장했습니다.")

### 7-6. 학습된 정책 평가하기

이제 학습된 DQN 정책을 **탐험 없이(greedy)** 실행해 보겠습니다.

이 단계에서는 두 가지를 확인합니다.

1. 평균적으로 얼마나 오래 버티는가?
2. 실제로 플레이 영상을 보면 어떻게 움직이는가?

수업에서는 이 결과를 통해  
**랜덤 정책 / 규칙 기반 정책 / DQN 정책의 차이**를 보여주면 좋습니다.

In [ ]:
def dqn_policy(obs):
    state_tensor = torch.tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    with torch.no_grad():
        q_values = dqn_model(state_tensor)
    return int(torch.argmax(q_values, dim=1).item())


def safe_evaluate(name, policy_fn, n_episodes=20, seed_start=0):
    try:
        rewards, lengths = evaluate_policy(policy_fn, n_episodes=n_episodes, seed_start=seed_start)
        summarize_result(name, rewards, lengths)
        return rewards, lengths
    except Exception as e:
        print(f"[{name}] 정책 평가를 건너뜁니다. 이유: {e}")
        return None, None


print("=== 정책 비교 (가능한 경우만 표시) ===")
_ = safe_evaluate("랜덤 정책", random_policy, n_episodes=20, seed_start=1000)
_ = safe_evaluate("각도만 사용 정책", rule_policy_angle_only, n_episodes=20, seed_start=1100)
_ = safe_evaluate("각도+각속도 정책", rule_policy_angle_velocity, n_episodes=20, seed_start=1200)
_ = safe_evaluate("나만의 규칙 정책", my_policy, n_episodes=20, seed_start=1300)
dqn_rewards, dqn_lengths = safe_evaluate("DQN 정책", dqn_policy, n_episodes=20, seed_start=1400)





### 7-7. 학습된 정책을 GIF로 저장하고 노트북에서 바로 보기

학습된 정책을 실제로 실행하면서 프레임을 모아 GIF를 저장합니다.  
이 GIF는 노트북 안에서 바로 렌더링할 수 있어, 수업 중 데모에 사용하기 좋습니다.

> 만약 실습 환경에서 프레임 수집이 잘 안 되면,  
> 이 셀은 건너뛰고 학습 곡선과 평균 성능만 보여주어도 충분합니다.

In [ ]:
def collect_frames_with_policy(policy_fn, max_steps=500, seed=2025):
    env = make_env(render_mode="rgb_array")
    obs, _ = reset_env(env, seed=seed)

    frames = []
    total_reward = 0.0

    # 초기 프레임
    try:
        frame = env.render()
        if frame is not None:
            frames.append(frame)
    except Exception:
        pass

    for step in range(max_steps):
        action = int(policy_fn(obs))
        next_obs, reward, done, _ = step_env(env, action)
        total_reward += reward
        obs = next_obs

        try:
            frame = env.render()
            if frame is not None:
                frames.append(frame)
        except Exception:
            pass

        if done:
            break

    env.close()
    return frames, total_reward, step + 1


frames, demo_reward, demo_length = collect_frames_with_policy(dqn_policy)

print(f"DQN 데모 1회 실행 결과: reward={demo_reward:.1f}, length={demo_length}")

gif_path = Path("cartpole_dqn_demo_cpu.gif")

if len(frames) > 0:
    imageio.mimsave(gif_path, frames, fps=30)
    display(Image(filename=str(gif_path)))
    print(f"GIF 저장 완료: {gif_path}")
else:
    print("프레임을 수집하지 못했습니다. render_mode 지원 여부를 확인해주세요.")

### 7-8. DQN 데모 해석

이번 데모에서 전달하고 싶은 핵심은 다음과 같습니다.

- 랜덤 정책은 거의 버티지 못한다
- 사람이 만든 규칙 기반 정책은 어느 정도 버틴다
- DQN은 상태를 입력으로 받아 행동가치를 학습하면서 더 좋은 정책을 찾을 수 있다

즉, 사람이 직접 규칙을 만드는 방식에서  
**학습으로 행동가치를 근사하는 방식으로 확장**되는 흐름을 보여주는 예시다.

#### 수업에서 강조하면 좋은 문장
- CartPole은 상태가 연속적이기 때문에 Q-table 방식이 어렵다
- 그래서 신경망으로 Q-value를 근사하는 DQN이 자연스럽게 등장한다
- 여기서는 구현 세부보다, **왜 Deep RL이 필요한지**를 이해하는 것이 핵심이다

## 8. 실습 정리

- CartPole은 상태, 행동, 보상으로 표현되는 강화학습 문제이다.
- 랜덤 정책보다 규칙 기반 정책이 더 좋은 성능을 낼 수 있다.
- 상태를 일부만 쓰는 것보다 더 적절히 사용하는 것이 성능에 영향을 줄 수 있다.
- 하지만 상태가 연속적이기 때문에 사람이 규칙을 직접 만들거나 Q-table로 모두 다루는 데 한계가 있다.
- 그래서 DQN 같은 Deep RL 방법이 등장한다.

## 체크 질문

1. CartPole의 상태에는 어떤 정보가 들어 있는가?
2. 막대 각도만 사용하는 정책의 장점과 한계는 무엇인가?
3. 각도와 각속도를 함께 쓰면 왜 더 나아질 수 있는가?
4. 사람이 직접 규칙을 만드는 방식의 한계는 무엇인가?
5. CartPole에서 DQN 같은 방법이 필요한 이유는 무엇인가?